In [1]:
import pandas as pd
from tortreinador.utils.preprocessing import load_data, ScalerConfig
from tortreinador.train import TorchTrainer, config_generator
from cVAE import cVAE, CombineLoss
from tortreinador.utils.View import init_weights, split_weights
import torch
from tortreinador.utils.metrics import r2_score

In [2]:
df_chunk_0 = pd.read_parquet(".\\data\\data_chunk_0.parquet")
df_chunk_1 = pd.read_parquet(".\\data\\data_chunk_1.parquet")

df_all = pd.concat([df_chunk_0, df_chunk_1])

In [3]:
input_parameters = [
    'Mass', 
    'Radius',
    'FeMg',
    'SiMg',
]


output_parameters = [
    'WRF',
    'MRF',
    'CRF',
    'WMF',
    'CMF', 
    'CPS',
    'CTP',
    'k2'
]

save_file_path = 'D:\\Resource\\MDN\\PackageTest\\resources'

In [4]:
t_loader, v_loader, test_x, test_y, s_x, s_y = load_data(data=df_all, input_parameters=input_parameters,
                                                         output_parameters=output_parameters, normal=ScalerConfig(on=True, method='minmax', feature_range=(0, 1), normal_y=True),
                                                         if_shuffle=True, batch_size=1024, n_workers=8, add_noise=True, error_rate=[0.14, 0.04, 0.12, 0.13], only_noise=True)

In [5]:
# Model
model = cVAE(i_dim=len(output_parameters), z_dim=int(len(output_parameters) * 7), num_hidden=1024, c_dim=len(input_parameters), o_dim=len(output_parameters))
init_weights(model)
criterion = CombineLoss()
optimizer = torch.optim.Adam(split_weights(model), lr=0.0008984, weight_decay=0.000001)

In [6]:
class Trainer(TorchTrainer):

    def calculate(self, x, y, mode='t'):
            
        recon_x, l_z_mu, l_z_logvar = self.model(y, x)

        loss, re_loss, kld = self.criterion(recon_x, y, l_z_mu, l_z_logvar, auto_adj=(0.02, 0.03))    # z, mu, logvar min(0.5 + self.current_epoch * 0.1, 1.5)
        metric_per = r2_score(y, recon_x)
        return self._standard_return(loss=loss, metric_per=metric_per, mode=mode, y=y)

In [7]:
trainer = Trainer(is_gpu=True, epoch=200, optimizer=optimizer, model=model, criterion=criterion, data_save_mode='csv')

Epoch:200, Device: cuda


In [8]:
config = config_generator(save_file_path, warmup_epochs=5, best_metric=0.5, auto_save=5, if_mix=True, initial_rate=0.7, final_rate=1.0, warmup_error=10, best_loss=0.07)

In [9]:
result = trainer.fit(t_loader, v_loader, **config)

D:\Anaconda\envs\DeepExo\lib\site-packages\tortreinador\utils\Recorder.py:102: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.checkpoint = torch.load(checkpoint_path)
Ep

KeyboardInterrupt: 